# Autgrad Engine - Comprehensive Test Notebook

This notebook demonstrates the complete functionality of the autgrad engine, built from scratch with automatic differentiation support.

**Topics covered:**
1. Basic tensor operations and gradients
2. Building blocks (neurons, layers, MLP)
3. Training on simple datasets
4. Solving the XOR problem
5. Computational graph visualization

## Setup

Import the autgrad library and required dependencies.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from autgrad import Tensor, Neuron, Layer, MLP, draw_dot

print("Autgrad library imported successfully!")

## Part 1: Basic Tensor Operations

Let's start with simple operations to understand how the autograd system works.

### Addition and Multiplication

In [ ]:
# Create simple tensors
a = Tensor(2.0)
b = Tensor(3.0)

# Perform operations
c = a * b
d = c + a

print(f"Forward pass result: d = {d.data}")
print(f"Expected: (a * b) + a = (2 * 3) + 2 = 8")

# Backpropagation
d.backward()

print(f"\nAfter backward pass:")
print(f"Gradient of a: {a.grad}")
print(f"Gradient of b: {b.grad}")
print(f"Gradient of c: {c.grad}")

print(f"\nManual verification (using chain rule):")
print(f"dd/da = dc/da + dc/da*(dc/db*db/da) = 1 + 1*(b) = 1 + 1*3 = 4 ✓")
print(f"dd/db = dc/db*dd/dc = a*1 = 2 ✓")

### Power and Exponential Operations

In [ ]:
# Test power operations
a = Tensor(2.0)
b = Tensor(3.0)
c = a * b
d = c + 2**a  # Power operation: 2 raised to the power of a

print(f"Forward pass: d = (a * b) + 2^a")
print(f"d = (2 * 3) + 2^2 = 6 + 4 = {d.data}")

d.backward()

print(f"\nAfter backward pass:")
print(f"Gradient of a: {a.grad:.4f}")
print(f"Gradient of b: {b.grad:.4f}")

print(f"\nManual verification:")
print(f"dd/da = db/da + d(2^a)/da = b + ln(2)*2^a = 3 + ln(2)*4 = 3 + {np.log(2)*4:.4f} = {3 + np.log(2)*4:.4f} ✓")
print(f"dd/db = dc/db = a = {a.data} ✓")

### Broadcasting with Vectors

In [ ]:
# Test broadcasting
a = Tensor([1.0, 2.0, 3.0])
b = Tensor([4.0, 5.0, 6.0])
c = a + b

print(f"Vector addition:")
print(f"a = {a.data}")
print(f"b = {b.data}")
print(f"c = a + b = {c.data}")

# Now sum the result
loss = c.sum()
print(f"\nSum of c: {loss.data}")

loss.backward()
print(f"\nGradients:")
print(f"Gradient of a: {a.grad}")
print(f"Gradient of b: {b.grad}")

## Part 2: ReLU Activation Function

In [ ]:
# Test ReLU activation
x = Tensor([-2.0, -0.5, 0.0, 1.5, 3.0])
y = x.relu()

print(f"Input:  {x.data}")
print(f"ReLU output: {y.data}")
print(f"Expected: [0, 0, 0, 1.5, 3.0]")

# Backprop through ReLU
loss = y.sum()
loss.backward()

print(f"\nGradients after backward:")
print(f"x.grad = {x.grad}")
print(f"Expected: [0, 0, 0, 1, 1] (gradient is 1 where input > 0, else 0)")

## Part 3: Single Neuron

In [ ]:
# Create a single neuron
neuron = Neuron(nin=3, activation='linear')

# Create input
x = Tensor([1.0, 2.0, 3.0])

# Forward pass
output = neuron(x)

print(f"Neuron architecture: {3} inputs -> 1 output (linear activation)")
print(f"Weight shape: {neuron.w.data.shape}")
print(f"Bias shape: {neuron.b.data.shape}")
print(f"\nInput: {x.data}")
print(f"Weights: {neuron.w.data}")
print(f"Bias: {neuron.b.data}")
print(f"Output: {output.data}")

# Training step
target = Tensor(1.0)
loss = (output - target) * (output - target)

neuron.zero_grad()
loss.backward()

print(f"\nLoss: {loss.data:.4f}")
print(f"Weight gradients: {neuron.w.grad}")
print(f"Bias gradient: {neuron.b.grad}")

## Part 4: Layer of Neurons

In [ ]:
# Create a layer with multiple neurons
layer = Layer(nin=3, nout=4, activation='relu')

# Input
x = Tensor([1.0, 2.0, 3.0])

# Forward pass
output = layer(x)

print(f"Layer architecture: {3} inputs -> {4} outputs (ReLU activation)")
print(f"Number of neurons: {len(layer.neurons)}")
print(f"Total parameters: {len(layer.parameters())}")
print(f"Output shape: {output.data.shape}")
print(f"Output values: {output.data}")

# Loss and backprop
target = Tensor([1.0, 2.0, 3.0, 4.0])
diff = output - target
loss = (diff * diff).sum()

layer.zero_grad()
loss.backward()

print(f"\nLoss: {loss.data:.4f}")
print(f"Gradients computed: {any(np.any(p.grad != 0) for p in layer.parameters())}")

## Part 5: Multi-Layer Perceptron (MLP)

In [ ]:
# Create an MLP: 3 inputs -> 4 hidden -> 4 hidden -> 1 output
model = MLP(nin=3, nouts=[4, 4, 1], activations=['relu', 'relu', 'linear'])

print(f"MLP Architecture:")
print(f"  Input: 3")
print(f"  Hidden layer 1: 4 neurons (ReLU)")
print(f"  Hidden layer 2: 4 neurons (ReLU)")
print(f"  Output layer: 1 neuron (Linear)")
print(f"\nTotal parameters: {len(model.parameters())}")

# Forward pass
x = Tensor([1.0, 2.0, 3.0])
output = model(x)

print(f"\nInput: {x.data}")
print(f"Output: {output.data}")

## Part 6: Training on a Simple Dataset

In [ ]:
# Create a simple dataset
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]  # Target labels

print(f"Dataset: {len(xs)} samples, 3 features per sample")
print(f"Samples: {xs}")
print(f"Targets: {ys}")

# Create model
model = MLP(3, [4, 4, 1], activations=['relu', 'relu', 'linear'])
learning_rate = 0.01

# Training loop
losses = []
for iteration in range(50):
    # Forward pass
    ypred = [model(Tensor(x)) for x in xs]
    ypred_tensor = Tensor.stack(ypred)
    ys_tensor = Tensor([[y] for y in ys])
    
    # Compute loss (MSE)
    diff = ypred_tensor + ys_tensor * Tensor(-1)
    loss = (diff * diff).sum()
    losses.append(loss.data)
    
    # Backward pass
    model.zero_grad()
    loss.backward()
    
    # Update weights
    for p in model.parameters():
        p.data += -learning_rate * p.grad
    
    if iteration % 10 == 0:
        print(f"Iteration {iteration}: Loss = {loss.data:.4f}")

print(f"\nFinal predictions: {ypred_tensor.data.flatten()}")
print(f"Target values:     {ys}")

In [ ]:
# Plot the loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(losses, linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss reduced from {losses[0]:.4f} to {losses[-1]:.4f}")

## Part 7: Solving the XOR Problem

The XOR problem is a classic challenge that requires a neural network with at least one hidden layer to solve.
XOR outputs 1 when inputs are different, and 0 when they're the same.

In [ ]:
# XOR dataset
xs = [[0, 0], [0, 1], [1, 0], [1, 1]]
ys = [-1, 1, 1, -1]  # -1 for 0, 1 for 1 in XOR

print("XOR Problem:")
print("Inputs\t| Output")
print("-" * 20)
for x, y in zip(xs, ys):
    output = 1 if y > 0 else 0
    print(f"{x[0]} XOR {x[1]}\t| {output}")

# Create model with sufficient capacity
# 2 inputs -> 4 hidden (ReLU) -> 4 hidden (ReLU) -> 1 output (Linear)
model = MLP(2, [4, 4, 1], activations=['relu', 'relu', 'linear'])

# Training parameters
learning_rate = 0.01
num_iterations = 400

# Training loop
losses = []
for iteration in range(num_iterations):
    # Forward pass
    ypred = [model(Tensor(x)) for x in xs]
    ypred_tensor = Tensor.stack(ypred)
    ys_tensor = Tensor([[y] for y in ys])
    
    # Compute loss (MSE)
    diff = ypred_tensor + ys_tensor * Tensor(-1)
    loss = (diff * diff).sum()
    losses.append(loss.data)
    
    # Backward pass
    model.zero_grad()
    loss.backward()
    
    # Update weights
    for p in model.parameters():
        p.data += -learning_rate * p.grad
    
    if iteration % 50 == 0:
        print(f"Iteration {iteration}: Loss = {loss.data:.4f}")

print(f"\nTraining completed!")
print(f"Final Loss: {losses[-1]:.4f}")

In [ ]:
# Evaluate on XOR problem
print("XOR Problem - Final Results:")
print("Inputs\t| Target | Predicted | Correct?")
print("-" * 50)

correct = 0
for x, y in zip(xs, ys):
    pred = model(Tensor(x)).data
    # Convert to -1/1 prediction
    pred_label = 1 if pred > 0 else -1
    is_correct = pred_label == y
    correct += int(is_correct)
    
    target_output = "1 (True)" if y > 0 else "0 (False)"
    pred_output = f"{pred:.4f}"
    correct_str = "✓" if is_correct else "✗"
    print(f"{x[0]} XOR {x[1]}\t| {target_output:12} | {pred_output:10} | {correct_str}")

accuracy = correct / len(xs) * 100
print(f"\nAccuracy: {accuracy:.1f}%")

In [ ]:
# Plot loss during training
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(losses, linewidth=2, color='steelblue')
plt.xlabel('Iteration')
plt.ylabel('Loss (MSE)')
plt.title('XOR Training - Loss Over Time')
plt.grid(True, alpha=0.3)

# Loss reduction
plt.subplot(1, 2, 2)
plt.semilogy(losses, linewidth=2, color='coral')
plt.xlabel('Iteration')
plt.ylabel('Loss (MSE, log scale)')
plt.title('XOR Training - Loss (Log Scale)')
plt.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
print(f"Loss reduction: {(1 - losses[-1]/losses[0])*100:.1f}%")

## Part 8: Computational Graph Visualization

Let's visualize the computational graph for a simple operation.

In [ ]:
# Create a simple computation graph
a = Tensor(2.0)
b = Tensor(3.0)
c = a * b
d = c + a

# Forward and backward
d.backward()

print(f"Computational graph for: d = (a * b) + a")
print(f"Forward:")
print(f"  a = {a.data}")
print(f"  b = {b.data}")
print(f"  c = a * b = {c.data}")
print(f"  d = c + a = {d.data}")
print(f"\nBackward (gradients):")
print(f"  d.grad = {d.grad}")
print(f"  c.grad = {c.grad}")
print(f"  a.grad = {a.grad}")
print(f"  b.grad = {b.grad}")

In [ ]:
# Try to visualize the graph (requires graphviz)
try:
    dot = draw_dot(d)
    dot.render('simple_graph', format='svg', cleanup=True)
    print("Graph visualization created!")
    print("Saved as 'simple_graph.svg'")
except Exception as e:
    print(f"Graph visualization not available: {e}")
    print("(Requires graphviz to be installed)")

## Part 9: Summary and Key Concepts

**What we've built:**

1. **Tensor class**: The core data structure that tracks:
   - Data (numpy array)
   - Gradients (derivatives w.r.t. loss)
   - Computational graph (parent relationships)

2. **Automatic Differentiation**: 
   - Forward pass: Compute outputs from inputs
   - Backward pass: Propagate gradients using chain rule

3. **Operations**: Support for +, -, *, /, **, and activation functions (ReLU)

4. **Neural Network Components**:
   - Neuron: Single unit with weights and bias
   - Layer: Collection of neurons
   - MLP: Stack of layers for deep learning

5. **Training**: Complete SGD optimization pipeline
   - Forward pass
   - Loss computation
   - Backward pass
   - Parameter updates

**Key insights:**
- The XOR problem requires non-linear transformations (ReLU)
- Gradients flow backward through the entire network
- Broadcasting allows operations on tensors of different shapes
- The topological sort ensures correct order of gradient computation